### Install & Import required libraries

In [1]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import faiss
import torch

### Define search query & top-k results to retrieve

In [ ]:
user_query = "An affordable hotel with a view of the Eiffel Tower"
top_k = 5

: 

### Load Dataset

In [ ]:
dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")

: 

extract the 'train' split and convert it into a pandas DataFrame

In [ ]:
df = dataset["train"].to_pandas()

: 

### Filter for Hotels only in Paris and clean the dataset from erroneous values

In [ ]:
df_paris = df.loc[
    (df.locality == "Paris") &
    (df.review_text.str.strip() != "") &
    (df.review_text.notna() )
].reset_index(drop=True)


: 

### Extract all reviews into a list

In [ ]:
reviews = df_paris.review_text.tolist()

: 

### Load an embedding model

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

: 

### move model to GPU (This is common in ML)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

: 

### Encode user search query & reviews

In [ ]:
query_embedding = model.encode([user_query])
review_embeddings = model.encode(reviews)

: 

### Make sure Embeddings are of type float32

In [ ]:
query_embedding = query_embedding.astype(np.float32)
review_embeddings = review_embeddings.astype(np.float32)

: 

### Normalize Embeddings

In [ ]:
faiss.normalize_L2(query_embedding)
faiss.normalize_L2(review_embeddings)

: 

### Check if dimensions of embeddings match

In [ ]:
assert query_embedding.shape[1] == review_embeddings.shape[1]
embedding_dimension = query_embedding.shape[1]

: 

## Perform Similarity Searching using FAISS

### Create an Index

In [ ]:
index = faiss.IndexFlatIP(embedding_dimension)

: 

### Add reviews

In [ ]:
index.add(review_embeddings)

: 

### Perform Similarity search

In [ ]:
similarity_scores, indices = index.search(query_embedding, top_k)

: 

### Output top-k most relevant results

In [ ]:
for i, (score, idx) in enumerate(zip(similarity_scores[0], indices[0]), start=1):
    row = df_paris.iloc[idx]
    
    print(f"Query: {user_query}")
    print(f"{i}. Hotel Name: {row.hotel_name} ")
    print(f"Review: {row.review_text}")
    print(f"Distance: {score}")
    print()

: 